In [1]:
from dotenv import load_dotenv, find_dotenv,get_key
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model='google/gemma-4-12b-qat',
        input=prompt
    )
    return response.output_text

In [4]:
question = 'I just discovered the course can I still join it?'
# answer = llm(question)
print(f'Q: {question}')
# print(f'A: {answer}')

Q: I just discovered the course can I still join it?


In [6]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
try:
    response = requests.get(docs_url)
    courses_raw = response.json()
    print(courses_raw)
except Exception as e:
    print(f'Error: {e}')



[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 115}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}]


In [7]:
documents = []
url_prefix = 'https://datatalks.club/faq/'

for course in courses_raw:
    course_url = url_prefix + course['path']
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

len(documents)
documents[1100]

{'id': '8351543816',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 4 Homework',
 'question': 'Homework: Why do I have different values of accuracy than the options in the homework?',
 'answer': 'One main reason behind this issue is the method of splitting the data. For example, if we want to split the data into train/validation/test with the ratios 60%/20%/20%, different methods may yield different results even if the final ratios are the same.\n\n1. Method 1:\n   \n   ```python\n   df_train, df_temp = train_test_split(df, test_size=0.4, random_state=42)\n   df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)\n   ```\n\n2. Method 2:\n   \n   ```python\n   df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)\n   df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)\n   ```\n\nWhile both methods achieve the same ratio, the data split differently, resulting in variations in accuracy. It is recomme

In [8]:
from minsearch import Index

index = Index(
  text_fields=['question', 'section' 'answer'],
  keyword_fields=['course']
)

index.fit(documents)

In [9]:
# index.search(question)
def search(question, course='llm-zoomcamp'):
  boost_dict={'question': 2.0}
  filter_dict={'course': course}

  return index.search(
    question, 
    boost_dict=boost_dict,
    filter_dict=filter_dict,
    num_results=5
)


In [10]:
INSTRUCTIONS = """
You are a helpful assistant that can answer questions about the course given the provided context.
Use the context to find relevant information and provide accurate answers.
If an answer is not found in the context, respond with "I don't know"
"""

USER_PROMPT_TEMPLATE = """
Question: {user_question}

Context:
{context}
"""

In [11]:
def build_context(search_results):
  lines = []

  for doc in search_results:
    lines.append(doc["section"])
    lines.append(f'Q: {doc["question"]}')
    lines.append(f'A: {doc["answer"]}')
    lines.append('')

  return '\n'.join(lines)

In [12]:
def build_prompt(question, search_results):
  context = build_context(search_results)
  prompt = USER_PROMPT_TEMPLATE.format(user_question=question, context=context)
  return prompt.strip()


search_results = search(question)
prompt = build_prompt(question, search_results)

In [13]:
def llm(system_instructions, prompt, model='gpt-5.4', reasoning='medium'):
    message_history = [
      { 'role': 'developer', 'content': system_instructions },
      { 'role': 'user', 'content': prompt }
    ]
    response = openai_client.responses.create(
        model=model,
        reasoning={"effort": reasoning},
        input=message_history
    )
    return response

In [14]:
response = llm(INSTRUCTIONS, prompt)


In [17]:
print(response)
print(response.usage)
print(response.output_text)

Response(id='resp_5d6895e426b8c7ae', created_at=1784171897.0, error=None, incomplete_details=None, instructions=None, metadata=None, model='gpt-5.4', object='response', output=[ResponseOutputMessage(id='msg_e5a5646024614e82', content=[ResponseOutputText(annotations=[], text='Yes — you can still join.\n\nIf you want to receive a certificate, you need to submit your project while submissions are still being accepted.', type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=None, temperature=None, tool_choice=None, tools=None, top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status='completed', text=None, top_logprobs=None, truncation=None, usage=ResponseUsage(input_tokens=570, input_tokens_de

In [ ]:
def rag(query, model='gpt-5.4', reasoning='medium'):
  search_results = search(query)
  prompt = build_prompt(query, search_results)
  answer = llm(INSTRUCTIONS, prompt, model, reasoning)
  return answer.output_text 